# Validate data cleaning and feature preparation

This notebook covers PBI 2 and PBI 3. It runs the same Python functions on the historical files and a synthetic practice batch, checks how those functions handle problem rows, and shows how to prepare training and evaluation data separately.

Use a Python kernel with `pandas` and `scikit-learn` installed. This notebook imports the reusable code in `src/`. The repository keeps the three historical CSVs in `data/raw/`. Set `SPBUY_DATA_DIR` to their folder if you store them elsewhere. Run the cells in order.

In [1]:
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
# VS Code may start the notebook kernel inside the notebooks folder.
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data_cleaning import (
    ORDER_KEY, clean_customers_data, clean_labels_data, clean_orders_data,
)
from src.features import (
    CATEGORICAL_FEATURES, NUMERIC_FEATURES, build_preprocessor,
    chronological_split, join_training_data, prepare_training_data,
)

cleaners = {
    "customers": clean_customers_data,
    "orders": clean_orders_data,
    "labels": clean_labels_data,
}
file_names = {name: f"{name}_base.csv" for name in cleaners}
data_override = os.environ.get("SPBUY_DATA_DIR")
candidate_dirs = (Path(data_override),) if data_override else (project_root / "data" / "raw",)
data_dir = next(
    (folder for folder in candidate_dirs if all((folder / name).is_file() for name in file_names.values())),
    None,
)
if data_dir is None:
    raise FileNotFoundError("Place all three *_base.csv files together or set SPBUY_DATA_DIR.")

raw = {
    name: pd.read_csv(data_dir / file_name, dtype="string")
    for name, file_name in file_names.items()
}

## Clean and review the historical data

Read the raw values as text to preserve identifiers and allow each cleaner to validate the original values. The cleaners standardise country codes, dates, booleans, and categorical text. Optional values that cannot be parsed stay missing. Required identifiers and labels are checked before rows can be accepted. Normalised duplicates are kept once; conflicting records are reported for review. Unusual but valid amounts are retained, including refunds needed for analytics.

The issue report records one row per finding, so its count can exceed the number of source rows removed. Check the accepted row counts separately from the issue totals.

In [2]:
cleaned, reports = {}, {}
for name, cleaner in cleaners.items():
    cleaned[name], reports[name] = cleaner(raw[name])

counts = pd.DataFrame.from_dict({
    name: {
        "input_rows": len(raw[name]),
        "accepted_rows": len(cleaned[name]),
        "rows_not_accepted": len(raw[name]) - len(cleaned[name]),
        "recorded_issues": len(reports[name]),
    }
    for name in cleaners
}, orient="index")
display(counts)

reasons = pd.concat(
    {name: report["reason"].value_counts() for name, report in reports.items()},
    names=["dataset", "reason"],
).rename("issue_count")
display(reasons.to_frame())

field_audit = pd.concat({
    name: pd.DataFrame({
        "missing_before": raw[name].isna().sum(),
        "missing_after": cleaned[name].isna().sum(),
        "cleaned_dtype": cleaned[name].dtypes.astype(str),
    })
    for name in cleaners
}, names=["dataset", "field"])
display(field_audit)

assert cleaned["customers"]["customer_id"].is_unique
for name in ("orders", "labels"):
    assert not cleaned[name].duplicated(ORDER_KEY).any()
assert cleaned["labels"]["is_fraud"].isin([0, 1]).all()
del raw

,input_rows,accepted_rows,rows_not_accepted,recorded_issues
customers,901312,901312,0,712
orders,949889,949889,0,221189
labels,950318,950318,0,0


issue_count
dataset   reason                           
customers missing_optional              712
orders    refund_exceeds_order       199687
          missing_optional            21502

missing_before  ...   cleaned_dtype
dataset   field                                             ...                
customers country_code                                   0  ...          string
          customer_id                                    0  ...          string
          mobile_verified                                0  ...         boolean
          num_orders_last_50days                         0  ...           Int64
          num_cancelled_orders_last_50days               0  ...           Int64
          num_refund_orders_last_50days                  0  ...           Int64
          total_payment_last_50days                      0  ...         Float64
          num_associated_customers                       0  ...           Int64
          first_order_datetime                         712  ...  datetime64[ns]
orders    country_code                                   0  ...          string
          order_id                                       0  ...          string
          collect_type                                   0  ...          string
          payment_method                             21501  ...          string
          order_value                                    0  ...         Float64
          num_items_ordered                              1  ...           Int64
          refund_value                                   0  ...         Float64
          order_date                                     0  ...  datetime64[ns]
labels    country_code                                   0  ...          string
          order_id                                       0  ...          string
          customer_id                                    0  ...          string
          is_fraud                                       0  ...            Int8

[21 rows x 3 columns]

## Join the tables and check coverage

The labels table links each order to a customer. Join orders to labels on `(country_code, order_id)` and then join customers on `(country_code, customer_id)`. Using country in both joins prevents a match to a record with the same ID in another country. The left joins preserve the orders and expose missing matches. A missing label remains unknown. A label without a corresponding order is counted separately.

Supervised training uses orders with a valid label, a matching customer, and a known order date. The date is required so each included order can be assigned to a chronological split. The joined table and the original cleaned tables remain available for other uses.

In [3]:
joined, audit = join_training_data(cleaned["customers"], cleaned["orders"], cleaned["labels"])
display(pd.Series(audit, name="rows", dtype="int64").to_frame())
assert len(joined) == len(cleaned["orders"])
assert not joined.duplicated(ORDER_KEY).any()

,rows
orders,949889
joined_orders,949889
matched_labels,949874
unmatched_labels,15
orphan_labels,444
matched_customers,949872
unmatched_customers,17
labelled_without_customer,2
orders_without_date,0
first_order_after_current_order,105


## Choose features available before the fraud decision

| Inputs | Why they are useful |
| --- | --- |
| Order value and item count | Describe the size of the basket. |
| Value per item | Describes basket value relative to item count. Leave it missing when the denominator is unavailable or zero. |
| Customer tenure in days | Describes how long the customer has been ordering. Compute from the first known order date and leave it missing when that date is absent or after the order being scored. |
| Country, collection type, and payment method | Describe order context without using raw identifiers. |
| Order weekday and month | Capture recurring calendar patterns. |

Keep `is_fraud` as the target. Exclude raw IDs and the current order's `refund_value` from the features. The customer snapshot columns, including the last 50 days' totals, mobile verification, and associated-customer count, have no observation timestamp in these files. Leave them out until the team can show their values were available at the decision time.

The evaluation set uses the latest eligible order dates. Dates are kept whole so one calendar day cannot appear on both sides. Fill missing numeric values with training medians and add indicators for missing fields observed during training. Use a log transform to reduce the influence of large values without dropping them, then scale numeric features. Encode each known category as a separate indicator. Fit the learned steps on training rows only, and leave the evaluation class balance as observed. Model owners can use class weights or training-only resampling if they choose.

In [4]:
X, y, metadata = prepare_training_data(joined)
train_index, evaluation_index = chronological_split(metadata)
assert X.index.equals(y.index) and X.index.equals(metadata.index)
assert set(X.columns) == set(NUMERIC_FEATURES) | set(CATEGORICAL_FEATURES)
assert {"is_fraud", "refund_value", "order_id", "customer_id"}.isdisjoint(X.columns)
assert not train_index.intersection(evaluation_index).size
assert len(train_index) + len(evaluation_index) == len(X)
assert metadata.loc[train_index, "order_date"].max() < metadata.loc[evaluation_index, "order_date"].min()

preprocessor = build_preprocessor()
X_train = preprocessor.fit_transform(X.loc[train_index])
X_evaluation = preprocessor.transform(X.loc[evaluation_index])
assert X_train.shape[0] == len(train_index)
assert X_evaluation.shape == (len(evaluation_index), X_train.shape[1])

split_summary = pd.DataFrame([
    {
        "split": name,
        "first_order_date": metadata.loc[rows, "order_date"].min().date(),
        "last_order_date": metadata.loc[rows, "order_date"].max().date(),
        "rows": len(rows),
        "fraud_pct": round(100 * float(y.loc[rows].mean()), 2),
        "transformed_columns": matrix.shape[1],
    }
    for name, rows, matrix in [
        ("train", train_index, X_train), ("evaluation", evaluation_index, X_evaluation),
    ]
])
display(split_summary)
del joined, X, y, metadata, X_train, X_evaluation

,split,first_order_date,last_order_date,rows,fraud_pct,transformed_columns
0,train,2022-11-12,2023-04-03,758025,10.60,43
1,evaluation,2023-04-04,2023-05-11,191847,12.54,43


## Run the same code on a synthetic practice batch

The batch deliberately includes three date formats, inconsistent category spelling and case, an exact duplicate, a conflicting order, an invalid target, a missing ID, and a customer from the wrong country. The same order ID also appears in two countries, which should produce two distinct orders. The zero-priced order and missing optional values exercise the feature rules without giving those rows a fraud label by default.

In [5]:
def practice_customer(country, customer, verified, first_order):
    return {
        "country_code": country, "customer_id": customer, "mobile_verified": verified,
        "num_orders_last_50days": 5, "num_cancelled_orders_last_50days": 0,
        "num_refund_orders_last_50days": 0, "total_payment_last_50days": 52,
        "num_associated_customers": 1, "first_order_datetime": first_order,
    }

def practice_order(country, order, date, value=20, items=2, payment="Credit card", refund=0):
    return {
        "country_code": country, "order_id": order, "collect_type": " Delivery ",
        "payment_method": payment, "order_value": value, "num_items_ordered": items,
        "refund_value": refund, "order_date": date,
    }

def practice_label(country, order, customer, fraud):
    return {
        "country_code": country, "order_id": order, "customer_id": customer, "is_fraud": fraud,
    }

first_order = practice_order(" my ", "shared-order", "2023/May/06", refund=25)
practice = {
    "customers": pd.DataFrame([
        practice_customer("my", "buyer-my", "yes", "2023-05-01 10:00:00"),
        practice_customer("PH", "buyer-ph", False, None),
    ]),
    "orders": pd.DataFrame([
        first_order, first_order.copy(),
        practice_order("ph", "shared-order", "2023-05-07", 0, None, None),
        practice_order("MY", "unlabelled", "2023/05/08"),
        practice_order("MY", "wrong-country", "2023-05-09"),
        practice_order("MY", "conflict", "2023-05-09", 10),
        practice_order("MY", "conflict", "2023-05-09", 30),
        practice_order("MY", "bad-target", "2023-05-10"),
        practice_order("MY", None, "2023-05-11"),
    ]),
    "labels": pd.DataFrame([
        practice_label("MY", "shared-order", "buyer-my", 1),
        practice_label("PH", "shared-order", "buyer-ph", 0),
        practice_label("MY", "wrong-country", "buyer-ph", 0),
        practice_label("MY", "bad-target", "buyer-my", 2),
        practice_label("MY", "no-order", "buyer-my", 1),
    ]),
}

practice_clean, practice_reports = {}, {}
for name, cleaner in cleaners.items():
    practice_clean[name], practice_reports[name] = cleaner(practice[name])
    rerun, _ = cleaner(practice_clean[name])
    pd.testing.assert_frame_equal(rerun, practice_clean[name])

assert len(practice_clean["orders"]) == 5
assert not practice_clean["orders"]["order_id"].eq("conflict").any()
assert not practice_clean["labels"]["order_id"].eq("bad-target").any()
assert practice_clean["orders"]["refund_value"].max() == 25
assert practice_clean["orders"]["num_items_ordered"].isna().sum() == 1
assert practice_clean["orders"]["order_date"].notna().all()
assert practice_clean["customers"].set_index("customer_id").loc["buyer-my", "mobile_verified"]
assert {"duplicate_record", "conflicting_key", "missing_required"}.issubset(practice_reports["orders"]["reason"])
assert practice_reports["labels"]["reason"].eq("invalid_fraud_label").any()
display(pd.concat(
    {name: report["reason"].value_counts() for name, report in practice_reports.items()},
    names=["dataset", "reason"],
).rename("issue_count").to_frame())

practice_joined, practice_audit = join_training_data(
    practice_clean["customers"], practice_clean["orders"], practice_clean["labels"],
)
practice_X, practice_y, practice_meta = prepare_training_data(practice_joined)
shared = practice_joined.loc[practice_joined["order_id"].eq("shared-order")].set_index("country_code")
assert shared["is_fraud"].to_dict() == {"MY": 1, "PH": 0}
assert len(practice_X) == 2 and set(practice_y) == {0, 1}
assert practice_X["value_per_item"].isna().sum() == 1
assert practice_X["customer_tenure_days"].dropna().tolist() == [5]
assert practice_joined.loc[practice_joined["order_id"].eq("bad-target"), "is_fraud"].isna().all()
display(pd.Series(practice_audit, name="rows", dtype="int64").to_frame())
print("Practice checks passed: keys, repeatable cleaning, missing values, and feature exclusions.")

issue_count
dataset   reason                           
customers missing_optional                1
orders    refund_exceeds_order            2
          missing_optional                2
          conflicting_key                 2
          duplicate_record                1
          missing_required                1
labels    invalid_fraud_label             1

,rows
orders,5
joined_orders,5
matched_labels,3
unmatched_labels,2
orphan_labels,1
matched_customers,2
unmatched_customers,3
labelled_without_customer,1
orders_without_date,0
first_order_after_current_order,0


Practice checks passed: keys, repeatable cleaning, missing values, and feature exclusions.


## Handover

For RDS loading, use the three accepted tables returned by `data_cleaning.py` and review the issue reports. Use `customer_id` as the customer identifier and `(country_code, order_id)` as the order/label key. Preserve `country_code` on all three tables and keep `refund_value` on orders for analytics. Confirm any foreign key rules against the reported unmatched records before loading. The cleaned dtype table above is the starting point for the SQL schema.

For retraining, import the same cleaners and functions in `features.py`. Assemble the accepted historical and incoming records, resolve conflicts across batches before joining, use the team's shared evaluation dates, and keep the fitted preprocessor with the model so future data uses the same transformations.